# Chapter 7, Guide 4: Simulating Pauli strings — the core routine

Companion notebook to *Quantum Algorithms from First Principles*, Chapter 7,
"Building a Pauli-sum simulator: the core routine for Chapters 8 and 9" and
"Testing a Pauli-sum simulator, and where it reappears".

The book asks for one function, `pauli_evolution(circuit, pauli_string, angle, qubits)`,
reused everywhere. The five steps: collect the non-identity positions; $H$ on $X$
positions and $S^\dagger$ then $H$ on $Y$ positions; a CNOT ladder into the last active
qubit; $R_z(2\theta)$ there; reverse. For a term $c_j P_j$ evolved for $\Delta t$, pass
$\theta = c_j\Delta t$.

Convention: a Pauli string is written in book order, leftmost letter on book wire 0
(top, most significant). `qubits` is the list of Qiskit qubits for book wires $0,1,\dots$

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, SparsePauliOp

bq = lambda j, tot: tot - 1 - j


def pauli_evolution(circuit, pauli_string, angle, qubits):
    """Append exp(-i * angle * P) to circuit, P the Pauli string on the given qubits.
    pauli_string[k] acts on qubits[k]. Implements the book's five steps."""
    active = [k for k, p in enumerate(pauli_string) if p != "I"]
    if not active:                      # pure identity: a global phase, skip
        return
    # step 2: basis changes
    for k in active:
        if pauli_string[k] == "X":
            circuit.h(qubits[k])
        elif pauli_string[k] == "Y":
            circuit.sdg(qubits[k]); circuit.h(qubits[k])
    # step 3: CNOT ladder into the last active qubit
    last = qubits[active[-1]]
    for k in active[:-1]:
        circuit.cx(qubits[k], last)
    # step 4: the rotation, with the factor of two
    circuit.rz(2 * angle, last)
    # step 5: reverse steps 3 and 2
    for k in reversed(active[:-1]):
        circuit.cx(qubits[k], last)
    for k in active:
        if pauli_string[k] == "X":
            circuit.h(qubits[k])
        elif pauli_string[k] == "Y":
            circuit.h(qubits[k]); circuit.s(qubits[k])


def pauli_matrix(pauli_string):
    """Dense matrix of a book-ordered Pauli string (leftmost letter = most significant)."""
    return SparsePauliOp(pauli_string).to_matrix()


def circuit_matrix(pauli_string, angle):
    n = len(pauli_string)
    qc = QuantumCircuit(n)
    pauli_evolution(qc, pauli_string, angle, [bq(j, n) for j in range(n)])
    return Operator(qc).data

## Test 1: single Pauli, $n=1$, against `expm`

In [2]:
theta = 0.37
for P in "XYZ":
    U = circuit_matrix(P, theta)
    E = expm(-1j * theta * pauli_matrix(P))
    assert np.allclose(U, E), P
print("Test 1 passed: exp(-i theta P) for P = X, Y, Z")

Test 1 passed: exp(-i theta P) for P = X, Y, Z


## Test 2: $Z\otimes Z$ gives the four phases $e^{\mp i\theta}$ by parity

In [3]:
U = circuit_matrix("ZZ", theta)
print("diagonal of exp(-i theta ZZ):", np.round(np.diag(U), 4))
print("expected phases:", np.round([np.exp(-1j * theta), np.exp(1j * theta), np.exp(1j * theta), np.exp(-1j * theta)], 4))
assert np.allclose(U, expm(-1j * theta * pauli_matrix("ZZ")))
print("Test 2 passed")

diagonal of exp(-i theta ZZ): [0.9323-0.3616j 0.9323+0.3616j 0.9323+0.3616j 0.9323-0.3616j]
expected phases: [0.9323-0.3616j 0.9323+0.3616j 0.9323+0.3616j 0.9323-0.3616j]
Test 2 passed


## Test 3: a weight-3 string containing $Y$ (catches $S$ versus $S^\dagger$)

In [4]:
for P in ["XYZ", "YYX", "ZYI", "IYY"]:
    assert np.allclose(circuit_matrix(P, theta), expm(-1j * theta * pauli_matrix(P))), P
# show that the wrong phase gate fails
qc = QuantumCircuit(1); qc.s(0); qc.h(0); qc.rz(2 * theta, 0); qc.h(0); qc.sdg(0)
print("S instead of S^dagger reproduces exp(-i theta Y)?", np.allclose(Operator(qc).data, expm(-1j * theta * pauli_matrix("Y"))))
print("Test 3 passed: weight-3 strings with Y are exact")

S instead of S^dagger reproduces exp(-i theta Y)? False
Test 3 passed: weight-3 strings with Y are exact


## Test 4: a non-commuting sum, error against $\Delta t$ with the expected slope

$H = 0.8\,XX + 0.5\,ZI + 0.3\,IZ$. The $Z$ terms commute with each other, so the
splitting is into two groups: $\{XX\}$ and $\{ZI, IZ\}$. One first-order step has error
$O(\Delta t^2)$ (slope 2); one symmetric step has error $O(\Delta t^3)$ (slope 3).

In [5]:
terms = [("XX", 0.8), ("ZI", 0.5), ("IZ", 0.3)]
H = sum(c * pauli_matrix(P) for P, c in terms)


def trotter_step(dt, symmetric):
    qc = QuantumCircuit(2); q = [bq(0, 2), bq(1, 2)]
    groupA, groupB = [terms[0]], terms[1:]
    if symmetric:
        for P, c in groupA: pauli_evolution(qc, P, c * dt / 2, q)
        for P, c in groupB: pauli_evolution(qc, P, c * dt, q)
        for P, c in groupA: pauli_evolution(qc, P, c * dt / 2, q)
    else:
        for P, c in groupA: pauli_evolution(qc, P, c * dt, q)
        for P, c in groupB: pauli_evolution(qc, P, c * dt, q)
    return Operator(qc).data


dts = np.array([0.4, 0.2, 0.1, 0.05, 0.025])
e1 = [np.linalg.norm(trotter_step(dt, False) - expm(-1j * H * dt), 2) for dt in dts]
e2 = [np.linalg.norm(trotter_step(dt, True) - expm(-1j * H * dt), 2) for dt in dts]
s1 = np.polyfit(np.log(dts), np.log(e1), 1)[0]
s2 = np.polyfit(np.log(dts), np.log(e2), 1)[0]
print(f"first-order single-step slope: {s1:.2f} (expect 2)")
print(f"symmetric single-step slope:   {s2:.2f} (expect 3)")
assert abs(s1 - 2) < 0.1 and abs(s2 - 3) < 0.15
plt.loglog(dts, e1, "o-", label="first order"); plt.loglog(dts, e2, "s-", label="symmetric")
plt.xlabel(r"$\Delta t$"); plt.ylabel("single-step error"); plt.legend(); plt.show()
print("Test 4 passed")

first-order single-step slope: 1.99 (expect 2)
symmetric single-step slope:   2.99 (expect 3)
Test 4 passed


## Ordering by qubit support lets CNOT ladders cancel

Two consecutive terms on the same support share a ladder; Qiskit's optimizer removes the
back-to-back CNOT pairs. Sorting by support is the cheap way to let it.

In [6]:
from qiskit import transpile
strings = ["ZZZI", "XXII", "ZZZI", "IZZZ", "XXII", "IZZZ"]   # three supports, interleaved
q4 = [bq(j, 4) for j in range(4)]
qc_unsorted = QuantumCircuit(4); qc_sorted = QuantumCircuit(4)
for P in strings: pauli_evolution(qc_unsorted, P, 0.1, q4)
for P in sorted(strings): pauli_evolution(qc_sorted, P, 0.1, q4)
for name, qc in [("unsorted", qc_unsorted), ("sorted by support", qc_sorted)]:
    t = transpile(qc, basis_gates=["u", "cx"], optimization_level=1)
    print(f"{name}: cx = {t.count_ops().get('cx', 0)}")

unsorted: cx = 20
sorted by support: cx = 10
